In [ ]:
import numpy as np
import pandas as pd
import polars as pl

In [ ]:
print("pandas version: ", pd.__version__)
print("polars version: ", pl.__version__)

## ダミーデータの作成

In [ ]:
def generate_dict_for_dataframe(n_rows=10000, seed=42, write_csv=None):
    """
    データフレーム作成用のダミーデータの辞書を返す。
    write_csvにパスを指定すると、そのパスにcsvとして出力する。
    """
    num_unique_customers = 2000
    
    # 乱数シードの固定
    np.random.seed(42)
    
    # 一般的な地域分類のリスト
    regions = ["北海道", "東北", "関東", "中部", "近畿", "中国", "四国", "九州"]
    
    # 顧客マスタデータの作成（ユニーク2000人分）
    unique_customer_ids = [f"C{i:04d}" for i in range(1, num_unique_customers + 1)]
    
    # 各顧客の一意な属性を生成
    genders_master = np.random.choice([1, 2], size=num_unique_customers)
    ages_master = np.random.randint(18, 81, size=num_unique_customers)
    regions_master = np.random.choice(regions, size=num_unique_customers)  # 地域分類マスタ
    
    # 契約年月日の生成
    start_contract = pd.to_datetime("2020-01-01")
    end_contract = pd.to_datetime("2024-12-31")
    contract_range = (end_contract - start_contract).days
    random_days_contract = np.random.randint(
        0, contract_range + 1, size=num_unique_customers
    )
    contract_dates_master = (
        (start_contract + pd.to_timedelta(random_days_contract, unit="D"))
        .strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )
    
    # マスタ辞書（顧客番号をキーに、各属性を固定）
    customer_master = {
        cid: {"性別": g, "年齢": a, "契約年月日": c, "出身地": r}
        for cid, g, a, c, r in zip(
            unique_customer_ids,
            genders_master,
            ages_master,
            contract_dates_master,
            regions_master,
        )
    }
    
    # 1万行の購買トランザクションデータの作成
    # 1万行分の顧客番号をランダムに割り当て
    customer_ids = np.random.choice(unique_customer_ids, size=n_rows)
    
    # 購買金額のベース生成
    purchase_amounts_base = np.random.randint(-5000, 50000, size=n_rows).astype(float)
    
    # 購買金額の1%（100行）をランダムに np.nan にする
    missing_amount_indices = np.random.choice(n_rows, size=int(n_rows * 0.01), replace=False)
    purchase_amounts = [
        np.nan if i in missing_amount_indices else amt
        for i, amt in enumerate(purchase_amounts_base)
    ]
    
    # 購買年月日（整数8桁）のベース生成
    start_date = pd.to_datetime("2025-01-01")
    end_date = pd.to_datetime("2025-12-31")
    date_range = (end_date - start_date).days
    random_days_purchase = np.random.randint(0, date_range + 1, size=n_rows)
    purchase_dates_base = (
        (start_date + pd.to_timedelta(random_days_purchase, unit="D"))
        .strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )
    
    # 購買年月日の1%（100行）をランダムに欠損値（None）にする
    missing_purchase_indices = np.random.choice(n_rows, size=int(n_rows * 0.01), replace=False)
    purchase_dates = [
        None if i in missing_purchase_indices else date
        for i, date in enumerate(purchase_dates_base)
    ]
    
    # 顧客番号をベースにマスタの属性を1万行に紐付け
    genders = [customer_master[cid]["性別"] for cid in customer_ids]
    ages = [customer_master[cid]["年齢"] for cid in customer_ids]
    contract_dates = [customer_master[cid]["契約年月日"] for cid in customer_ids]
    regions_base = [customer_master[cid]["出身地"] for cid in customer_ids]
    
    # 出身地（地域）の1%（100行）をランダムに欠損値（None）にする
    missing_region_indices = np.random.choice(n_rows, size=int(n_rows * 0.01), replace=False)
    birthplaces = [
        None if i in missing_region_indices else rg for i, rg in enumerate(regions_base)
    ]
    
    # pandasに渡せる辞書データの作成
    dummy_data_dict = {
        "顧客番号": customer_ids,
        "契約年月日": contract_dates,
        "年齢": ages,
        "性別": genders,
        "出身地": birthplaces,
        "購入年月日": purchase_dates,
        "購買金額": purchase_amounts,
    }
        
    if write_csv:
        pl.DataFrame(dummy_data_dict).write_csv(write_csv) # デフォルトの１万行で約500KB
    elif not write_csv:
        return dummy_data_dict

## 1. データフレームの作成

In [ ]:
# 辞書型のデータの雛形
dummy_data_dict = generate_dict_for_dataframe(n_rows=10000, seed=42)

In [ ]:
# pandas
df_pd = pd.DataFrame(dummy_data_dict)
print(df_pd)

In [ ]:
# polars
df_pl = pl.DataFrame(dummy_data_dict)
print(df_pl)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html
# print(help(pd.DataFrame))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/dataframe/index.html
# print(help(pl.DataFrame))

## 2. データフレームの作成: csv読み込み

In [ ]:
# csvファイルの生成
generate_dict_for_dataframe(n_rows=10000, seed=42, write_csv="dummy_data.csv")

In [ ]:
# pandas
df_pd = pd.read_csv("dummy_data.csv")
print(df_pd)

In [ ]:
# polars
df_pl = pl.read_csv("dummy_data.csv")
print(df_pl)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# print(help(pd.read_csv))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
# print(help(pl.read_csv))

## 3. インデックス列の追加

In [ ]:
df_pl = pl.DataFrame(dummy_data_dict).with_row_index()
print(df_pl)

In [ ]:
# https://docs.pola.rs/api/python/version/0.20/reference/dataframe/api/polars.DataFrame.with_row_index.html
# print(help(pl.DataFrame.with_row_index))

## 4. 列の追加

In [ ]:
# pandas
df_pd.assign(
    集計用=1, 
    出身地方=df_pd["出身地"] + "地方",
)

In [ ]:
# polars
df_pl.with_columns(
    pl.lit(1).alias("集計用"),
    (pl.col("出身地") + "地方").alias("出身地方"),
)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html
# print(help(pd.DataFrame.assign))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html
# print(help(pl.DataFrame.with_columns))

## 5. 列名の変更

In [ ]:
# pandas
df_pd.rename(columns={"購買金額": "利用額"})

In [ ]:
# polars
df_pl.rename({"購買金額": "利用額"})

In [ ]:
# polars pl.colを用いた列名変更(推奨)
df_pl.with_columns(
    pl.col("購買金額").alias("利用額") # .aliasで新しい列名を指定
)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
# print(help(pd.DataFrame.rename))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.rename.html
# print(help(pl.DataFrame.rename))

## 6. 列の選択

In [ ]:
# pandas
df_pd.filter(items=["性別", "顧客番号"])

In [ ]:
# polars
df_pl.select(
    pl.col("性別"), 
    pl.col("顧客番号"),
)

In [ ]:
# polars
# こういう書き方もできる
df_pl.select(["性別", "顧客番号"])
df_pl[["性別", "顧客番号"]]

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html
# print(help(pd.DataFrame.filter))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.select.html
# print(help(pl.select))

## 7. 列の削除

In [ ]:
# pandas
df_pd.drop(columns=["性別"])

In [ ]:
# polars
df_pl.select(pl.exclude("列"))

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
# print(help(pd.DataFrame.drop))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.exclude.html
# print(help(pl.exclude))

## 8. 行の絞り込み

In [ ]:
# pandas
df_pd.query('性別 == 1 & 年齢 <= 50')

# ↓ query関数を使わない形
# df_pd[
#     (df_pd["性別"] == 1) & (df_pd["年齢"] <= 50)
# ]

In [ ]:
# polars
df_pl.filter(
    (pl.col("性別") == 1) & (pl.col("年齢") <= 50)
)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html
# print(help(pd.DataFrame.query))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.filter.html
# print(help(pl.DataFrame.filter))

## 9. 重複の削除

In [ ]:
# pandas
df_pd.drop_duplicates(subset=["年齢"])

In [ ]:
# polars
df_pl.unique(subset=pl.col("年齢"), keep="first", maintain_order=True)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html
# print(help(pd.DataFrame.drop_duplicates))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.unique.html
# print(help(pl.DataFrame.unique))

## 11. 欠損値の検出

In [ ]:
# pandas
df_pd.isnull().sum()

In [ ]:
# polars: is_null どのデータ型に対しても利用できる
df_pl.select(
    pl.all().is_null()
).sum()

In [ ]:
# polars: is_nan 数値型に対してのみ利用できる
import polars.selectors as cs
df_pl.select(
    cs.numeric().is_nan()
).sum()

In [ ]:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isnull.html
# print(help(pd.DataFrame.isnull))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.is_null.html
# print(help(pl.Expr.is_null))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.is_nan.html
# print(help(pl.Expr.is_nan))

## 12. 欠損値の削除

In [ ]:
# pandas
df_pd.dropna()

In [ ]:
# polars (drop_nulls)
df_pl.drop_nulls()

In [ ]:
# polars (drop_nulls)
df_pl.drop_nulls()

In [ ]:
# polars (drop_nans)
df_pl.drop_nans()

In [ ]:
# polars (drop_nulls + drop_nans) pandasのdropnaと同じ
df_pl.drop_nulls().drop_nans()

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html
# help(pd.DataFrame.dropna)

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.drop_nulls.html
# help(pl.DataFrame.drop_nulls)

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.drop_nans.html
# help(pl.DataFrame.drop_nans)

## 13. 欠損値の値埋め

In [ ]:
# pandas
df_pd.fillna(0)

In [ ]:
# polars (fill_null)
df_pl.fill_null(0)

In [ ]:
# polars (fill_nan)
df_pl.fill_nan(0)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html
# print(help(pd.DataFrame.fillna))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.fill_null.html
# print(help(pl.DataFrame.fill_null))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.fill_nan.html
# print(help(pl.DataFrame.fill_nan))

## 14. データフレームの結合: 縦方向

In [ ]:
# 結合用の新しいデータフレームの用意
dummy_dict_data2 = generate_dict_for_dataframe(seed=1)
df_pd2 = pd.DataFrame(dummy_dict_data2)
df_pl2 = pl.DataFrame(dummy_dict_data2).with_row_index()

In [ ]:
# pandas
pd.concat([df_pd, df_pd2])

In [ ]:
# polars
pl.concat([df_pl, df_pl2])

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.concat.html
# print(help(pd.concat))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/api/polars.concat.html
# print(help(pl.concat))

## 15. データフレームの結合: 横方向

In [ ]:
# pandas
pd.merge(df_pd, df_pd2, on="顧客番号", how="left")

In [ ]:
# polars
df_pl.join(df_pl2, on="顧客番号", how="left")

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.merge.html
# print(help(pd.merge))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.join.html
# print(help(pl.DataFrame.join))

## 16. 並び替え

In [ ]:
# pandas
df_pd.sort_values(
    ["年齢", "購買金額"], 
    ascending=[True, False],
    na_position="first",
)

In [ ]:
# polars
df_pl.sort(
    [pl.col("年齢"), pl.col("購買金額")], 
    descending=[False, True],
    maintain_order=True,
    nulls_last=True,
)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html
# print(help(pd.DataFrame.sort_values))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.sort.html
# print(help(pl.DataFrame.sort))

## 17. グループバイ

In [ ]:
# pandas
df_pd.groupby("顧客番号", as_index=False, sort=False).agg(
    年齢=("年齢", "last"),
    契約年月日=("契約年月日", "last"),
    合計購買金額=("購買金額", "sum"),
)

In [ ]:
# polars
df_pl.group_by(
    pl.col("顧客番号"),
    maintain_order=True
).agg(
    pl.last("年齢"),
    pl.last("契約年月日"),
    pl.sum("購買金額").alias("合計購入金額"),
)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
# print(help(pd.DataFrame.groupby))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.group_by.html
# print(help(pl.DataFrame.group_by))

## 18. 記述統計

In [ ]:
# pandas
df_pd.describe()

In [ ]:
# polars
df_pl.describe()

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html
# print(help(pd.DataFrame.describe))

In [ ]:
# https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.describe.html
# print(help(pl.DataFrame.describe))

## 19. 関数の適用

In [ ]:
# 適用する関数
def label_sex(sex):
    if sex == 1: return "男性"
    elif sex == 2: return "女性"

In [ ]:
# pandas
df_pd.assign(
    性別_男女=df_pd["性別"].apply(label_sex)
)

In [ ]:
# polars
df_pl.with_columns(
    pl.col("性別").map_elements(label_sex, return_dtype=pl.String).alias("性別_男女")
)

In [ ]:
# できるだけpolarsの既存の表現で処理できるようにする
df_pl.with_columns(
    pl.when(pl.col("性別") == 1).then(pl.lit("男性"))
      .when(pl.col("性別") == 2).then(pl.lit("女性"))
    .alias("性別_男女")
)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html
# print(help(pd.DataFrame.apply))

In [ ]:
# https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.map_elements.html
# print(help(pl.Expr.map_elements))